# CELL 1 — Install packages

In [1]:
!pip install networkx python-louvain matplotlib seaborn pandas numpy -q
print("✅ Packages installed")

✅ Packages installed


# CELL 2 — Mount Drive + Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import community as community_louvain
import ast, os, re, time, warnings
warnings.filterwarnings('ignore')

# ── Paths
DRIVE_PATH  = '/content/drive/MyDrive/PoliGraphX'
OUTPUT_PATH = os.path.join(DRIVE_PATH, 'outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("✅ Imports done")


Mounted at /content/drive
✅ Imports done


# CELL 3 — Load dataset

In [3]:
print("Loading tweets_sentiment_labeled.csv...")

df = pd.read_csv(os.path.join(DRIVE_PATH, 'tweets_sentiment_labeled.csv'), low_memory=False)
df = df[df['clean_text'].notna()].copy()
df = df.reset_index(drop=True)

print(f"Total tweets : {len(df):,}")
print(f"Columns      : {list(df.columns)}")
print(f"\nIdeology distribution:")
print(df['ideology'].value_counts())

# ── Build user → ideology map
user_ideology = df.groupby('username')['ideology'].agg(
    lambda x: x.value_counts().index[0]
).to_dict()
print(f"\nUnique users : {len(user_ideology):,}")
print("✅ Data loaded")

Loading tweets_sentiment_labeled.csv...
Total tweets : 42,110
Columns      : ['text', 'rawContent', 'date', 'lang', 'username', 'likeCount', 'retweetCount', 'replyCount', 'quoteCount', 'viewCount', 'hashtags', 'mentionedUsers', 'retweetedTweet', 'epoch', 'id_str', 'location', 'tweet_text', 'date_only', 'year', 'month', 'day', 'hour', 'hashtags_list', 'hashtags_str', 'hashtag_count', 'mentioned_list', 'mention_count', 'clean_text', 'ideology', 'ideology_confidence', 'vader_sentiment', 'vader_confidence', 'textblob_sentiment', 'textblob_confidence', 'roberta_sentiment', 'roberta_confidence']

Ideology distribution:
ideology
Right    26866
Left     15244
Name: count, dtype: int64

Unique users : 28,200
✅ Data loaded


# CELL 4 — Build edges from Mentions

In [4]:
print("Building edges from mentions...")

mention_edges = []

for _, row in df.iterrows():
    source = row['username']
    if pd.isna(source):
        continue

    # Parse mentionedUsers column
    mentions = row.get('mentionedUsers', None)
    if pd.isna(mentions) or mentions == '' or mentions is None:
        continue

    try:
        # Handle both string list and actual list
        if isinstance(mentions, str):
            # Try to extract usernames from string
            users = re.findall(r"'username':\s*'([^']+)'", mentions)
            if not users:
                users = re.findall(r'"username":\s*"([^"]+)"', mentions)
            if not users:
                # Plain comma separated
                users = [u.strip().strip("'\"[]") for u in mentions.split(',')]
        elif isinstance(mentions, list):
            users = [u.get('username', '') if isinstance(u, dict) else str(u) for u in mentions]
        else:
            users = []

        for target in users:
            target = str(target).strip()
            if target and target != source and target != 'nan':
                mention_edges.append((source, target, 'mention'))
    except Exception:
        continue

print(f"Mention edges found : {len(mention_edges):,}")

Building edges from mentions...
Mention edges found : 294,051


# CELL 5 — Build edges from Retweets

In [5]:
print("Building edges from retweets...")

retweet_edges = []

for _, row in df.iterrows():
    source = row['username']
    if pd.isna(source):
        continue

    rt = row.get('retweetedTweet', None)
    if pd.isna(rt) or rt == '' or rt is None:
        continue

    try:
        if isinstance(rt, str):
            users = re.findall(r"'username':\s*'([^']+)'", rt)
            if not users:
                users = re.findall(r'"username":\s*"([^"]+)"', rt)
        elif isinstance(rt, dict):
            users = [rt.get('username', '')]
        else:
            users = []

        for target in users:
            target = str(target).strip()
            if target and target != source and target != 'nan':
                retweet_edges.append((source, target, 'retweet'))
    except Exception:
        continue

print(f"Retweet edges found : {len(retweet_edges):,}")

all_edges = mention_edges + retweet_edges
print(f"Total edges         : {len(all_edges):,}")



Building edges from retweets...
Retweet edges found : 0
Total edges         : 294,051


# CELL 6 — Build NetworkX Graph

In [6]:
print("Building NetworkX graph...")

G = nx.DiGraph()

# Add edges with type
for source, target, etype in all_edges:
    if G.has_edge(source, target):
        G[source][target]['weight'] += 1
    else:
        G.add_edge(source, target, weight=1, type=etype)

# Add node ideology attributes
for node in G.nodes():
    G.nodes[node]['ideology'] = user_ideology.get(node, 'Unknown')

print(f"Graph built:")
print(f"  Nodes : {G.number_of_nodes():,}")
print(f"  Edges : {G.number_of_edges():,}")

# Keep only the largest connected component for visualization
G_undirected = G.to_undirected()
largest_cc   = max(nx.connected_components(G_undirected), key=len)
G_sub        = G.subgraph(largest_cc).copy()

print(f"\nLargest connected component:")
print(f"  Nodes : {G_sub.number_of_nodes():,}")
print(f"  Edges : {G_sub.number_of_edges():,}")
print("✅ Graph built")

Building NetworkX graph...
Graph built:
  Nodes : 75,371
  Edges : 247,693

Largest connected component:
  Nodes : 75,371
  Edges : 247,693
✅ Graph built


# CELL 7 — Compute Network Metrics

In [7]:
print("Computing network metrics...")

# Degree centrality
in_degree  = dict(G.in_degree(weight='weight'))
out_degree = dict(G.out_degree(weight='weight'))
degree     = dict(G.degree(weight='weight'))

# PageRank (most influential users)
pagerank = nx.pagerank(G, alpha=0.85, max_iter=200)

# Store metrics
metrics_df = pd.DataFrame({
    'username'   : list(G.nodes()),
    'ideology'   : [user_ideology.get(n, 'Unknown') for n in G.nodes()],
    'in_degree'  : [in_degree.get(n, 0) for n in G.nodes()],
    'out_degree' : [out_degree.get(n, 0) for n in G.nodes()],
    'degree'     : [degree.get(n, 0) for n in G.nodes()],
    'pagerank'   : [pagerank.get(n, 0) for n in G.nodes()],
})

metrics_df = metrics_df.sort_values('pagerank', ascending=False).reset_index(drop=True)

print("\nTop 10 Most Influential Users (PageRank):")
print(metrics_df[['username', 'ideology', 'pagerank', 'in_degree']].head(10).to_string(index=False))

metrics_path = os.path.join(OUTPUT_PATH, 'network_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f"\n✅ Metrics saved → {metrics_path}")

Computing network metrics...

Top 10 Most Influential Users (PageRank):
    username ideology  pagerank  in_degree
indices': [0  Unknown  0.030715      31080
        13]}  Unknown  0.004957       4840
        16]}  Unknown  0.003884       4167
        12]}  Unknown  0.003842       3884
        15]}  Unknown  0.003671       3597
        10]}  Unknown  0.003488       3211
        14]}  Unknown  0.003046       3292
        11]}  Unknown  0.002901       3127
         9]}  Unknown  0.002042       2318
         8]}  Unknown  0.001877       1719

✅ Metrics saved → /content/drive/MyDrive/PoliGraphX/outputs/network_metrics.csv


# CELL 8 — Community Detection (Louvain)

In [8]:
print("Running Community Detection (Louvain)...")

G_louvain   = G_sub.to_undirected()
partition   = community_louvain.best_partition(G_louvain, random_state=42)
modularity  = community_louvain.modularity(partition, G_louvain)

# Add community to nodes
nx.set_node_attributes(G_sub, partition, 'community')

# Analyze communities
comm_df = pd.DataFrame({
    'username'  : list(partition.keys()),
    'community' : list(partition.values()),
    'ideology'  : [user_ideology.get(n, 'Unknown') for n in partition.keys()]
})

print(f"Modularity score : {modularity:.4f}")
print(f"Number of communities : {comm_df['community'].nunique()}")
print(f"\nTop communities by size:")
print(comm_df['community'].value_counts().head(10))

# Check ideology composition of top communities
print("\nIdeology composition of top 5 communities:")
top_comms = comm_df['community'].value_counts().head(5).index
for c in top_comms:
    sub = comm_df[comm_df['community'] == c]
    left  = (sub['ideology'] == 'Left').sum()
    right = (sub['ideology'] == 'Right').sum()
    print(f"  Community {c}: {len(sub)} users | Left={left} | Right={right}")

print("✅ Community detection complete")

Running Community Detection (Louvain)...
Modularity score : 0.5731
Number of communities : 54

Top communities by size:
community
1     8786
2     6120
11    4874
5     4280
0     4242
10    3948
3     3925
6     3888
9     3641
15    3426
Name: count, dtype: int64

Ideology composition of top 5 communities:
  Community 1: 8786 users | Left=1131 | Right=2374
  Community 2: 6120 users | Left=224 | Right=385
  Community 11: 4874 users | Left=514 | Right=995
  Community 5: 4280 users | Left=508 | Right=764
  Community 0: 4242 users | Left=606 | Right=1343
✅ Community detection complete


# CELL 9 — Echo Chamber Detection

In [10]:
print(f"Same ideology  : {same_ideology}")
print(f"Cross ideology : {cross_ideology}")
print(f"Unknown        : {unknown}")
print(f"Total edges    : {G.number_of_edges()}")

# Check how many nodes have known ideology
known = sum(1 for n in G.nodes() if user_ideology.get(n, 'Unknown') != 'Unknown')
print(f"\nNodes with known ideology : {known}")
print(f"Total nodes               : {G.number_of_nodes()}")

Same ideology  : 0
Cross ideology : 0
Unknown        : 247693
Total edges    : 247693

Nodes with known ideology : 22100
Total nodes               : 75371


In [12]:
# Build reverse ideology map
# If unknown user is mostly mentioned by Left → assign Left, etc.
print("Assigning ideology to unknown nodes...")

for node in G.nodes():
    if user_ideology.get(node, 'Unknown') != 'Unknown':
        continue

    # Look at who mentions this node
    predecessors = list(G.predecessors(node))
    if not predecessors:
        continue

    ideologies = [user_ideology.get(p, 'Unknown') for p in predecessors]
    ideologies = [i for i in ideologies if i != 'Unknown']

    if not ideologies:
        continue

    # Assign majority ideology
    from collections import Counter
    most_common = Counter(ideologies).most_common(1)[0][0]
    user_ideology[node] = most_common

# Update node attributes
for node in G.nodes():
    G.nodes[node]['ideology'] = user_ideology.get(node, 'Unknown')

# Check again
known = sum(1 for n in G.nodes() if user_ideology.get(n, 'Unknown') != 'Unknown')
print(f"Nodes with known ideology after fix : {known}")
print(f"Total nodes                         : {G.number_of_nodes()}")

Assigning ideology to unknown nodes...
Nodes with known ideology after fix : 75371
Total nodes                         : 75371


In [13]:
print("Analyzing Echo Chambers...")

# Count cross-ideology vs same-ideology interactions
same_ideology  = 0
cross_ideology = 0
unknown        = 0

for source, target, data in G.edges(data=True):
    src_ideo = user_ideology.get(source, 'Unknown')
    tgt_ideo = user_ideology.get(target, 'Unknown')

    if src_ideo == 'Unknown' or tgt_ideo == 'Unknown':
        unknown += 1
    elif src_ideo == tgt_ideo:
        same_ideology += 1
    else:
        cross_ideology += 1

total_known = same_ideology + cross_ideology
echo_score  = same_ideology / total_known if total_known > 0 else 0

print(f"\nEcho Chamber Analysis:")
print(f"  Same-ideology interactions  : {same_ideology:,} ({same_ideology/total_known*100:.1f}%)")
print(f"  Cross-ideology interactions : {cross_ideology:,} ({cross_ideology/total_known*100:.1f}%)")
print(f"  Unknown                     : {unknown:,}")
print(f"\n  Echo Chamber Score : {echo_score:.4f}")
print(f"  (Score > 0.6 = Strong Echo Chamber | < 0.5 = Mixed)")

# Left ↔ Right interaction matrix
left_to_left   = sum(1 for s,t,_ in G.edges(data=True)
                     if user_ideology.get(s)=='Left'  and user_ideology.get(t)=='Left')
left_to_right  = sum(1 for s,t,_ in G.edges(data=True)
                     if user_ideology.get(s)=='Left'  and user_ideology.get(t)=='Right')
right_to_left  = sum(1 for s,t,_ in G.edges(data=True)
                     if user_ideology.get(s)=='Right' and user_ideology.get(t)=='Left')
right_to_right = sum(1 for s,t,_ in G.edges(data=True)
                     if user_ideology.get(s)=='Right' and user_ideology.get(t)=='Right')

interaction_matrix = pd.DataFrame(
    [[left_to_left, left_to_right],
     [right_to_left, right_to_right]],
    index=['Left Source', 'Right Source'],
    columns=['Left Target', 'Right Target']
)
print(f"\nInteraction Matrix:")
print(interaction_matrix)
print("✅ Echo chamber analysis complete")

Analyzing Echo Chambers...

Echo Chamber Analysis:
  Same-ideology interactions  : 185,447 (74.9%)
  Cross-ideology interactions : 62,246 (25.1%)
  Unknown                     : 0

  Echo Chamber Score : 0.7487
  (Score > 0.6 = Strong Echo Chamber | < 0.5 = Mixed)

Interaction Matrix:
              Left Target  Right Target
Left Source         20324         57511
Right Source         4735        165123
✅ Echo chamber analysis complete


# CELL 10 — Graph 1: Left vs Right Interaction Network

In [14]:
print("Generating Graph 1 — Left vs Right Network...")

# Limit to top N nodes for clean visualization
TOP_N  = 300
top_nodes = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)
top_nodes = [n for n, _ in top_nodes if n in G_sub.nodes()][:TOP_N]
G_vis     = G_sub.subgraph(top_nodes).copy()

# Node colors by ideology
color_map = {'Left': '#2196F3', 'Right': '#FF5722', 'Unknown': '#9E9E9E'}
node_colors = [color_map.get(G_vis.nodes[n].get('ideology', 'Unknown'), '#9E9E9E')
               for n in G_vis.nodes()]

# Node sizes by pagerank
pr_values  = [pagerank.get(n, 0) for n in G_vis.nodes()]
max_pr     = max(pr_values) if pr_values else 1
node_sizes = [300 + (pr / max_pr) * 2000 for pr in pr_values]

fig, ax = plt.subplots(1, 1, figsize=(16, 14))
fig.patch.set_facecolor('#0D0D0D')
ax.set_facecolor('#0D0D0D')

pos = nx.spring_layout(G_vis, k=0.5, iterations=50, seed=42)

# Draw edges
nx.draw_networkx_edges(G_vis, pos, ax=ax,
                       alpha=0.15, edge_color='#FFFFFF',
                       width=0.5, arrows=False)

# Draw nodes
nx.draw_networkx_nodes(G_vis, pos, ax=ax,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.9)

# Labels for top 20 only
top20 = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)
top20 = [n for n, _ in top20 if n in G_vis.nodes()][:20]
labels = {n: n for n in top20}
nx.draw_networkx_labels(G_vis, pos, labels, ax=ax,
                        font_size=7, font_color='white', font_weight='bold')

# Legend
legend_elements = [
    mpatches.Patch(color='#2196F3', label='Left'),
    mpatches.Patch(color='#FF5722', label='Right'),
    mpatches.Patch(color='#9E9E9E', label='Unknown'),
]
ax.legend(handles=legend_elements, loc='upper left',
          fontsize=12, facecolor='#1A1A1A', labelcolor='white',
          edgecolor='white')

ax.set_title('Political Interaction Network\n(Node size = PageRank influence | Color = Ideology)',
             fontsize=14, fontweight='bold', color='white', pad=20)
ax.axis('off')

plt.tight_layout()
g1_path = os.path.join(OUTPUT_PATH, 'graph7_interaction_network.png')
plt.savefig(g1_path, dpi=150, bbox_inches='tight', facecolor='#0D0D0D')
plt.show()
print(f"✅ Graph 1 saved → {g1_path}")

Generating Graph 1 — Left vs Right Network...
✅ Graph 1 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph7_interaction_network.png


# CELL 11 — Graph 2: Echo Chamber Visualization

In [15]:
print("Generating Graph 2 — Echo Chamber Analysis...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Echo Chamber Analysis', fontsize=15, fontweight='bold')

# ── Left plot: Same vs Cross ideology pie
ax1 = axes[0]
sizes  = [same_ideology, cross_ideology]
labels = [f'Same Ideology\n{same_ideology:,} ({same_ideology/total_known*100:.1f}%)',
          f'Cross Ideology\n{cross_ideology:,} ({cross_ideology/total_known*100:.1f}%)']
colors_pie = ['#FF9800', '#4CAF50']
wedges, texts = ax1.pie(sizes, labels=labels, colors=colors_pie,
                         startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for text in texts:
    text.set_fontsize(11)
ax1.set_title(f'Same vs Cross Ideology Interactions\n(Echo Score: {echo_score:.3f})',
              fontsize=13, fontweight='bold')

# ── Right plot: Interaction heatmap
ax2 = axes[1]
matrix_data = np.array([[left_to_left, left_to_right],
                          [right_to_left, right_to_right]])
sns.heatmap(matrix_data,
            annot=True, fmt=',d', cmap='RdYlGn',
            xticklabels=['→ Left', '→ Right'],
            yticklabels=['Left →', 'Right →'],
            ax=ax2, linewidths=1, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'})
ax2.set_title('Interaction Matrix\n(Who talks to whom?)',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Target Ideology', fontsize=11)
ax2.set_ylabel('Source Ideology', fontsize=11)

plt.tight_layout()
g2_path = os.path.join(OUTPUT_PATH, 'graph8_echo_chamber.png')
plt.savefig(g2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Graph 2 saved → {g2_path}")

Generating Graph 2 — Echo Chamber Analysis...
✅ Graph 2 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph8_echo_chamber.png


# CELL 12 — Graph 3: Top Influential Users

In [16]:
print("Generating Graph 3 — Top Influential Users...")

top_left  = metrics_df[metrics_df['ideology'] == 'Left'].head(10)
top_right = metrics_df[metrics_df['ideology'] == 'Right'].head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Top 10 Most Influential Users by Ideology\n(Ranked by PageRank)',
             fontsize=15, fontweight='bold')

for ax, data, title, color in zip(
    axes,
    [top_left, top_right],
    ['Top Left Users', 'Top Right Users'],
    ['#2196F3', '#FF5722']
):
    bars = ax.barh(data['username'], data['pagerank'],
                   color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, data['pagerank']):
        ax.text(bar.get_width() + 0.00001,
                bar.get_y() + bar.get_height()/2,
                f'{val:.5f}', va='center', fontsize=9)

    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('PageRank Score', fontsize=11)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
g3_path = os.path.join(OUTPUT_PATH, 'graph9_top_users.png')
plt.savefig(g3_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Graph 3 saved → {g3_path}")

Generating Graph 3 — Top Influential Users...
✅ Graph 3 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph9_top_users.png


# CELL 13 — Graph 4: Community Clustering

In [17]:
print("Generating Graph 4 — Community Clustering...")

# Color top 6 communities, rest = grey
top6_comms   = comm_df['community'].value_counts().head(6).index.tolist()
comm_palette = ['#E91E63', '#9C27B0', '#2196F3', '#FF9800', '#4CAF50', '#00BCD4']
comm_color_map = {c: comm_palette[i] for i, c in enumerate(top6_comms)}

G_comm = G_sub.subgraph(top_nodes).copy()
node_comm_colors = []
for n in G_comm.nodes():
    c = partition.get(n, -1)
    node_comm_colors.append(comm_color_map.get(c, '#9E9E9E'))

fig, ax = plt.subplots(figsize=(16, 14))
fig.patch.set_facecolor('#0D0D0D')
ax.set_facecolor('#0D0D0D')

pos2 = nx.spring_layout(G_comm, k=0.5, iterations=50, seed=99)

nx.draw_networkx_edges(G_comm, pos2, ax=ax,
                       alpha=0.1, edge_color='white',
                       width=0.5, arrows=False)
nx.draw_networkx_nodes(G_comm, pos2, ax=ax,
                       node_color=node_comm_colors,
                       node_size=node_sizes[:len(G_comm.nodes())],
                       alpha=0.9)

legend_elements2 = [mpatches.Patch(color=comm_palette[i],
                    label=f'Community {c}') for i, c in enumerate(top6_comms)]
legend_elements2.append(mpatches.Patch(color='#9E9E9E', label='Other'))
ax.legend(handles=legend_elements2, loc='upper left',
          fontsize=11, facecolor='#1A1A1A', labelcolor='white', edgecolor='white')

ax.set_title(f'Community Clustering (Louvain)\nModularity = {modularity:.4f}',
             fontsize=14, fontweight='bold', color='white', pad=20)
ax.axis('off')

plt.tight_layout()
g4_path = os.path.join(OUTPUT_PATH, 'graph10_communities.png')
plt.savefig(g4_path, dpi=150, bbox_inches='tight', facecolor='#0D0D0D')
plt.show()
print(f"✅ Graph 4 saved → {g4_path}")

Generating Graph 4 — Community Clustering...
✅ Graph 4 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph10_communities.png


# CELL 14 — Save Network Summary Report

In [18]:
summary = {
    'Total Nodes'               : G.number_of_nodes(),
    'Total Edges'               : G.number_of_edges(),
    'Largest Component Nodes'   : G_sub.number_of_nodes(),
    'Largest Component Edges'   : G_sub.number_of_edges(),
    'Communities Detected'      : comm_df['community'].nunique(),
    'Modularity Score'          : round(modularity, 4),
    'Echo Chamber Score'        : round(echo_score, 4),
    'Same Ideology Interactions': same_ideology,
    'Cross Ideology Interactions': cross_ideology,
    'Left to Left'              : left_to_left,
    'Left to Right'             : left_to_right,
    'Right to Left'             : right_to_left,
    'Right to Right'            : right_to_right,
}

summary_df   = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_path = os.path.join(OUTPUT_PATH, 'network_summary.csv')
summary_df.to_csv(summary_path, index=False)

print("Network Summary Report:")
print(summary_df.to_string(index=False))
print(f"\n✅ Saved → {summary_path}")


Network Summary Report:
                     Metric       Value
                Total Nodes  75371.0000
                Total Edges 247693.0000
    Largest Component Nodes  75371.0000
    Largest Component Edges 247693.0000
       Communities Detected     54.0000
           Modularity Score      0.5731
         Echo Chamber Score      0.7487
 Same Ideology Interactions 185447.0000
Cross Ideology Interactions  62246.0000
               Left to Left  20324.0000
              Left to Right  57511.0000
              Right to Left   4735.0000
             Right to Right 165123.0000

✅ Saved → /content/drive/MyDrive/PoliGraphX/outputs/network_summary.csv


# CELL 15 — Final Summary

In [19]:
print("=" * 55)
print("NOTEBOOK 4 COMPLETE ✅")
print("=" * 55)
print(f"\nOutputs saved to: {OUTPUT_PATH}")
print(f"  ✅ graph7_interaction_network.png")
print(f"  ✅ graph8_echo_chamber.png")
print(f"  ✅ graph9_top_users.png")
print(f"  ✅ graph10_communities.png")
print(f"  ✅ network_metrics.csv")
print(f"  ✅ network_summary.csv")
print(f"\n{'='*55}")
print(f"  Ready for Final Step — Combined Analysis & Report")
print(f"{'='*55}")

NOTEBOOK 4 COMPLETE ✅

Outputs saved to: /content/drive/MyDrive/PoliGraphX/outputs
  ✅ graph7_interaction_network.png
  ✅ graph8_echo_chamber.png
  ✅ graph9_top_users.png
  ✅ graph10_communities.png
  ✅ network_metrics.csv
  ✅ network_summary.csv

  Ready for Final Step — Combined Analysis & Report
